In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:

# ============================================================
# CELL 10: PREPROCESSING CONFIGURATION + SMOKE TEST
# ============================================================

from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path(
    "/kaggle/input/competitions/"
    "tlvmc-parkinsons-freezing-gait-prediction"
)

ACC_COLS = ["AccV", "AccML", "AccAP"]
G_TO_MS2 = 9.80665

COHORT_CONFIG = {
    "tdcsfog": {
        "fs": 128,
        "acc_unit": "m/s2",
        "label_cols": ["StartHesitation", "Turn", "Walking"],
        "mask_cols": [],
    },
    "defog": {
        "fs": 100,
        "acc_unit": "g",
        "label_cols": ["StartHesitation", "Turn", "Walking"],
        "mask_cols": ["Valid", "Task"],
    },
    "notype": {
        "fs": 100,
        "acc_unit": "g",
        "label_cols": ["Event"],
        "mask_cols": ["Valid", "Task"],
    },
}


def preprocess_recording(path, cohort):
    """
    Normalize one labeled recording.

    Output:
      - time_s: time in seconds
      - AccV, AccML, AccAP: acceleration expressed in g
      - fog_any: generic FoG indicator
      - annotation_valid: whether the row is eligible for
        supervised training under the current masking rules
      - fog_onset: start of a contiguous positive label run
      - subject to be attached later from verified metadata
    """
    if cohort not in COHORT_CONFIG:
        raise ValueError(f"Unknown cohort: {cohort}")

    cfg = COHORT_CONFIG[cohort]
    df = pd.read_csv(path)

    required = {"Time", *ACC_COLS, *cfg["label_cols"], *cfg["mask_cols"]}
    missing = required - set(df.columns)

    if missing:
        raise ValueError(
            f"{Path(path).name} ({cohort}) is missing: {sorted(missing)}"
        )

    if df.empty:
        raise ValueError(f"Empty recording: {path}")

    # Verify timestamps before using them.
    time_values = df["Time"].to_numpy()

    if not np.isfinite(time_values).all():
        raise ValueError(f"Non-finite timestamps: {path}")

    if np.any(np.diff(time_values) <= 0):
        raise ValueError(f"Time is not strictly increasing: {path}")

    # Normalize all accelerometer axes to g.
    acc = df[ACC_COLS].to_numpy(dtype=np.float64)

    if not np.isfinite(acc).all():
        raise ValueError(f"Missing/non-finite acceleration: {path}")

    if cfg["acc_unit"] == "m/s2":
        acc = acc / G_TO_MS2

    out = pd.DataFrame({
        "time_s": time_values / cfg["fs"],
        "AccV": acc[:, 0],
        "AccML": acc[:, 1],
        "AccAP": acc[:, 2],
    })

    # Standardize cohort-specific labels to one FoG indicator.
    labels = df[cfg["label_cols"]].fillna(0).to_numpy()

    if not np.isin(labels, [0, 1]).all():
        raise ValueError(f"Unexpected non-binary labels: {path}")

    out["fog_any"] = labels.max(axis=1).astype(np.uint8)

    # tDCSFOG recordings have no Valid/Task columns.
    # For DEFOG and notype, use only explicitly valid task rows.
    if cfg["mask_cols"]:
        annotation_valid = np.ones(len(df), dtype=bool)

        for col in cfg["mask_cols"]:
            annotation_valid &= df[col].fillna(False).to_numpy(dtype=bool)
    else:
        annotation_valid = np.ones(len(df), dtype=bool)

    out["annotation_valid"] = annotation_valid

    # Do not interpret masked rows as confirmed negative examples.
    out["fog_any"] = out["fog_any"].where(annotation_valid, np.nan)

    # Onset = first positive sample after a non-positive sample.
    # Mask boundaries are treated as boundaries, not as negative labels.
    valid = out["annotation_valid"].to_numpy()
    fog = out["fog_any"].fillna(0).to_numpy(dtype=np.uint8)

    previous_valid = np.r_[False, valid[:-1]]
    previous_fog = np.r_[0, fog[:-1]]

    onset = valid & (fog == 1) & (
        (~previous_valid) | (previous_fog == 0)
    )

    out["fog_onset"] = onset.astype(np.uint8)

    return out


# Smoke-test one recording from each labeled cohort.
for cohort in ["tdcsfog", "defog", "notype"]:
    paths = sorted((ROOT / "train" / cohort).glob("*.csv"))

    if not paths:
        raise FileNotFoundError(
            f"No training CSV files found for {cohort}: "
            f"{ROOT / 'train' / cohort}"
        )

    sample_path = paths[0]
    processed = preprocess_recording(sample_path, cohort)

    valid_rows = processed["annotation_valid"]
    valid_labels = processed.loc[valid_rows, "fog_any"]

    print(f"\n{'=' * 55}")
    print(f"Cohort: {cohort}")
    print(f"Sample: {sample_path.name}")
    print(f"Rows: {len(processed):,}")
    print(f"Sampling rate: {COHORT_CONFIG[cohort]['fs']} Hz")
    print("Acceleration unit after preprocessing: g")
    print(f"Valid annotation rows: {valid_rows.sum():,}")
    print(f"Valid annotation coverage: {valid_rows.mean():.2%}")
    print(f"FoG-positive valid rows: {(valid_labels == 1).sum():,}")
    print(f"FoG-onset samples: {processed['fog_onset'].sum():,}")
    print(
        "Acceleration ranges (g):\n",
        processed[ACC_COLS].agg(["min", "max"]).round(4)
    )

    assert np.isfinite(processed[ACC_COLS].to_numpy()).all()
    assert processed["time_s"].is_monotonic_increasing
    assert processed["fog_onset"].isin([0, 1]).all()

print("\nPreprocessing smoke tests completed.")

In [ ]:

# ============================================================
# CELL 11: VALIDATE LABELS AGAINST EVENT METADATA
# ============================================================

events_path = ROOT / "events.csv"
events = pd.read_csv(events_path)

print("Events metadata columns:", events.columns.tolist())
display(events.head())

def audit_label_events(cohort, recording_id):
    path = ROOT / "train" / cohort / f"{recording_id}.csv"
    raw = pd.read_csv(path)
    processed = preprocess_recording(path, cohort)

    label_cols = COHORT_CONFIG[cohort]["label_cols"]
    raw_positive = raw[label_cols].fillna(0).max(axis=1).astype(bool)

    print(f"\n--- {cohort}: {recording_id} ---")
    print("Rows:", len(raw))
    print("Raw positive rows:", int(raw_positive.sum()))
    print(
        "Positive rows outside eligible mask:",
        int((raw_positive & ~processed["annotation_valid"]).sum())
    )
    print(
        "Positive rows inside eligible mask:",
        int((raw_positive & processed["annotation_valid"]).sum())
    )
    print("Detected onset samples:", int(processed["fog_onset"].sum()))

    # Inspect event metadata only where this ID is present.
    if "Id" in events.columns:
        event_rows = events[events["Id"].astype(str) == recording_id]
        print("Matching event metadata rows:", len(event_rows))
        if len(event_rows):
            display(event_rows.head(10))
    else:
        print("No 'Id' column found in events.csv; inspect its schema above.")


# Use one representative file per cohort.
audit_label_events("tdcsfog", "003f117e14")
audit_label_events("defog", "02ea782681")
audit_label_events("notype", "02ab235146")

In [ ]:

# ============================================================
# CELL 12: EVENT TIMING ALIGNMENT AUDIT
# ============================================================

def compare_event_timing(cohort, recording_id):
    path = ROOT / "train" / cohort / f"{recording_id}.csv"

    raw = pd.read_csv(path)
    processed = preprocess_recording(path, cohort)

    ev = events[events["Id"].astype(str) == recording_id].copy()
    ev = ev.sort_values("Init").reset_index(drop=True)

    onset_times = processed.loc[
        processed["fog_onset"].eq(1), "time_s"
    ].to_numpy()

    # Positive-label interval boundaries, restricted to eligible rows.
    valid_positive = (
        processed["annotation_valid"]
        & processed["fog_any"].eq(1)
    )

    positive = processed.loc[valid_positive, "time_s"].to_numpy()

    print(f"\n{'=' * 70}")
    print(f"{cohort} | {recording_id}")
    print(f"Event rows: {len(ev)}")
    print(f"Detected label onsets: {len(onset_times)}")

    if ev.empty:
        print("No event metadata rows found.")
        return

    # Match each event Init to the nearest detected onset.
    # This is diagnostic only; it does not alter the labels.
    rows = []

    for _, event in ev.iterrows():
        init = float(event["Init"])
        completion = float(event["Completion"])

        if len(onset_times):
            nearest_idx = int(np.argmin(np.abs(onset_times - init)))
            nearest_onset = float(onset_times[nearest_idx])
            onset_error = nearest_onset - init
        else:
            nearest_onset = np.nan
            onset_error = np.nan

        # Find positive samples within the annotated event interval.
        in_interval = positive[
            (positive >= init - 1 / COHORT_CONFIG[cohort]["fs"])
            & (positive <= completion + 1 / COHORT_CONFIG[cohort]["fs"])
        ]

        rows.append({
            "Init_s": init,
            "Completion_s": completion,
            "Duration_s": completion - init,
            "Nearest_label_onset_s": nearest_onset,
            "Onset_error_s": onset_error,
            "Positive_samples_in_event": len(in_interval),
            "Type": event["Type"],
        })

    result = pd.DataFrame(rows)
    display(result.round(4))

    print("\nAbsolute onset error summary (seconds):")
    print(result["Onset_error_s"].abs().describe().round(4))

    print(
        "\nEvents with positive samples in annotated interval:",
        int((result["Positive_samples_in_event"] > 0).sum()),
        "/", len(result)
    )


compare_event_timing("tdcsfog", "003f117e14")
compare_event_timing("defog", "02ea782681")
compare_event_timing("notype", "02ab235146")

In [ ]:

# ============================================================
# CELL 13: FULL TRAINING EVENT-ALIGNMENT AUDIT
# ============================================================

from scipy.optimize import linear_sum_assignment

audit_rows = []

for cohort in ["tdcsfog", "defog", "notype"]:
    folder = ROOT / "train" / cohort
    paths = sorted(folder.glob("*.csv"))

    for i, path in enumerate(paths, start=1):
        recording_id = path.stem

        # Process labels and onset transitions.
        proc = preprocess_recording(path, cohort)
        detected = proc.loc[
            proc["fog_onset"].eq(1), "time_s"
        ].to_numpy()

        # Event metadata is keyed by recording ID.
        ev = events.loc[
            events["Id"].astype(str).eq(recording_id)
        ].sort_values("Init")

        event_init = ev["Init"].to_numpy(dtype=float)

        # Match detected onsets to event starts one-to-one.
        if len(detected) and len(event_init):
            cost = np.abs(
                event_init[:, None] - detected[None, :]
            )
            event_idx, onset_idx = linear_sum_assignment(cost)
            errors = cost[event_idx, onset_idx]
        else:
            errors = np.array([], dtype=float)

        audit_rows.append({
            "cohort": cohort,
            "recording_id": recording_id,
            "rows": len(proc),
            "event_count": len(event_init),
            "detected_onset_count": len(detected),
            "count_match": len(event_init) == len(detected),
            "matched_pairs": len(errors),
            "unmatched_events": len(event_init) - len(errors),
            "unmatched_onsets": len(detected) - len(errors),
            "mean_abs_error_s": (
                float(errors.mean()) if len(errors) else np.nan
            ),
            "max_abs_error_s": (
                float(errors.max()) if len(errors) else np.nan
            ),
        })

    print(f"Finished {cohort}: {len(paths)} recordings")

event_audit_df = pd.DataFrame(audit_rows)

print("\nSUMMARY BY COHORT")
display(
    event_audit_df.groupby("cohort").agg(
        recordings=("recording_id", "count"),
        count_match_rate=("count_match", "mean"),
        event_count=("event_count", "sum"),
        detected_onsets=("detected_onset_count", "sum"),
        recordings_with_mismatches=(
            "count_match", lambda x: int((~x).sum())
        ),
        mean_abs_error_s=("mean_abs_error_s", "mean"),
        max_abs_error_s=("max_abs_error_s", "max"),
    ).round(5)
)

print("\nFIRST 30 RECORDINGS WITH COUNT MISMATCHES")
mismatches = event_audit_df.loc[
    ~event_audit_df["count_match"]
]
display(mismatches.head(30))

print("\nLargest onset timing errors")
display(
    event_audit_df.sort_values(
        "max_abs_error_s", ascending=False
    ).head(20)
)

# Save the compact audit output for reproducibility.
audit_output = Path("event_alignment_audit.csv")
event_audit_df.to_csv(audit_output, index=False)
print(f"\nSaved: {audit_output.resolve()}")

In [ ]:

# CELL 14: DIAGNOSE EVENT-TO-LABEL COUNT MISMATCHES

import numpy as np
import pandas as pd

diagnostic_rows = []

mismatch_records = event_audit_df.loc[
    ~event_audit_df["count_match"],
    ["cohort", "recording_id", "event_count", "detected_onset_count"]
]

for rec in mismatch_records.itertuples(index=False):
    cohort = rec.cohort
    recording_id = rec.recording_id

    path = ROOT / "train" / cohort / f"{recording_id}.csv"
    proc = preprocess_recording(path, cohort)

    ev = (
        events.loc[events["Id"].astype(str).eq(recording_id)]
        .sort_values(["Init", "Completion"])
        .reset_index(drop=True)
    )

    times = proc["time_s"].to_numpy()
    labels = proc["fog_any"].to_numpy()
    onsets = proc.loc[
        proc["fog_onset"].eq(1), "time_s"
    ].to_numpy()

    for j, event in ev.iterrows():
        start = float(event["Init"])
        end = float(event["Completion"])

        # Label coverage inside this annotated event interval
        interval_mask = (times >= start) & (times <= end)
        interval_labels = labels[interval_mask]
        positive_samples = int(np.nansum(interval_labels == 1))

        # Is there a distinct label onset near this event's start?
        nearby_onsets = onsets[
            np.abs(onsets - start) <= 0.05
        ]
        nearest_onset_error = (
            float(np.min(np.abs(nearby_onsets - start)))
            if len(nearby_onsets)
            else np.nan
        )

        # Relationship to the previous annotated event
        if j > 0:
            previous_end = float(ev.loc[j - 1, "Completion"])
            gap_from_previous_s = start - previous_end
        else:
            gap_from_previous_s = np.nan

        diagnostic_rows.append({
            "cohort": cohort,
            "recording_id": recording_id,
            "event_index": j,
            "event_start_s": start,
            "event_end_s": end,
            "event_type": event.get("Type", np.nan),
            "positive_samples_in_event": positive_samples,
            "distinct_onset_within_50ms": len(nearby_onsets) > 0,
            "nearest_onset_error_s": nearest_onset_error,
            "gap_from_previous_event_s": gap_from_previous_s,
        })

event_diagnostic_df = pd.DataFrame(diagnostic_rows)

print("EVENT-LEVEL DIAGNOSTIC SUMMARY")
display(
    event_diagnostic_df.groupby("cohort").agg(
        events_examined=("event_index", "size"),
        events_with_positive_labels=(
            "positive_samples_in_event",
            lambda x: int((x > 0).sum())
        ),
        events_without_positive_labels=(
            "positive_samples_in_event",
            lambda x: int((x == 0).sum())
        ),
        events_with_distinct_onset=(
            "distinct_onset_within_50ms", "sum"
        ),
    )
)

print("\nEVENTS WITH NO DISTINCT LABEL ONSET")
display(
    event_diagnostic_df.loc[
        ~event_diagnostic_df["distinct_onset_within_50ms"]
    ].head(40)
)

print("\nEVENTS OVERLAPPING OR VERY CLOSE TO PREVIOUS EVENT")
display(
    event_diagnostic_df.loc[
        event_diagnostic_df["gap_from_previous_event_s"].between(
            -0.05, 0.10
        )
    ].head(40)
)

event_diagnostic_df.to_csv(
    "event_label_mismatch_diagnostic.csv", index=False
)
print("\nSaved: event_label_mismatch_diagnostic.csv")

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

# ---------------------------------------------------------
# 1. Select the recording and load its event annotations
# ---------------------------------------------------------
recording_id = "10e83d6550"
recording_path = ROOT / "train" / "tdcsfog" / f"{recording_id}.csv"

# If your notebook uses a different folder structure:
if not recording_path.exists():
    matches = list(ROOT.rglob(f"{recording_id}.csv"))
    if not matches:
        raise FileNotFoundError(f"Could not find {recording_id}.csv under {ROOT}")
    recording_path = matches[0]

df = pd.read_csv(recording_path)
events = pd.read_csv(ROOT / "events.csv")
events = events[events["Id"].astype(str) == recording_id].copy()
events = events.sort_values(["Init", "Completion"]).reset_index(drop=True)

print("Recording:", recording_path)
print("Recording shape:", df.shape)
print("Recording columns:", df.columns.tolist())
print("\nEvents for this recording:")
display(events)

# ---------------------------------------------------------
# 2. Inspect sample-level labels around each event boundary
# ---------------------------------------------------------
FS = COHORT_CONFIG["tdcsfog"]["fs"]
typed_cols = ["StartHesitation", "Turn", "Walking"]
missing = [c for c in typed_cols if c not in df.columns]
if missing:
    raise ValueError(f"Expected label columns not found: {missing}")

# Use the same sample-time convention as preprocessing.
df["time_s"] = df["Time"] / FS
df["fog_any"] = df[typed_cols].max(axis=1)

# Examine 1 second before and after each event start.
HALF_WINDOW_S = 1.0

for i, event in events.iterrows():
    boundary = float(event["Init"])
    left = df[df["time_s"].between(
        boundary - HALF_WINDOW_S, boundary, inclusive="left"
    )]
    right = df[df["time_s"].between(
        boundary, boundary + HALF_WINDOW_S, inclusive="both"
    )]

    print("\n" + "=" * 80)
    print(
        f"Event {i}: type={event.get('Type')} | "
        f"Init={event['Init']:.3f}s | "
        f"Completion={event['Completion']:.3f}s"
    )

    def label_summary(part):
        if part.empty:
            return {"rows": 0}
        return {
            "rows": len(part),
            "StartHesitation_positive": int(part["StartHesitation"].sum()),
            "Turn_positive": int(part["Turn"].sum()),
            "Walking_positive": int(part["Walking"].sum()),
            "fog_any_positive": int(part["fog_any"].sum()),
            "fog_any_first": int(part["fog_any"].iloc[0]),
            "fog_any_last": int(part["fog_any"].iloc[-1]),
        }

    print("Before boundary:", label_summary(left))
    print("After boundary: ", label_summary(right))

    # Show a small sample around the boundary.
    around = df[
        df["time_s"].between(
            boundary - 0.15, boundary + 0.15, inclusive="both"
        )
    ].copy()

    cols = ["Time", "time_s", *typed_cols, "fog_any"]
    display(around[cols].head(12))
    display(around[cols].tail(12))

# ---------------------------------------------------------
# 3. Find transitions in each typed label and in fog_any
# ---------------------------------------------------------
print("\n" + "=" * 80)
print("Label transitions within the recording")

for col in [*typed_cols, "fog_any"]:
    values = df[col].fillna(0).astype(int)
    change_idx = np.flatnonzero(values.to_numpy()[1:] != values.to_numpy()[:-1]) + 1

    transitions = pd.DataFrame({
        "sample_index": change_idx,
        "time_s": df.loc[change_idx, "time_s"].to_numpy(),
        "previous_value": values.iloc[change_idx - 1].to_numpy(),
        "new_value": values.iloc[change_idx].to_numpy(),
    })

    print(f"\n{col}: {len(transitions)} transitions")
    display(transitions.head(30))

# ---------------------------------------------------------
# 4. Compare event starts with nearby combined-label onsets
# ---------------------------------------------------------
fog = df["fog_any"].fillna(0).astype(int).to_numpy()
onset_idx = np.flatnonzero((fog[1:] == 1) & (fog[:-1] == 0)) + 1
onset_times = df.loc[onset_idx, "time_s"].to_numpy()

MATCH_TOLERANCE_S = 0.05
comparison = []

for _, event in events.iterrows():
    init = float(event["Init"])
    nearest = (
        float(onset_times[np.argmin(np.abs(onset_times - init))])
        if len(onset_times) else np.nan
    )
    error = abs(nearest - init) if np.isfinite(nearest) else np.nan

    comparison.append({
        "event_type": event.get("Type"),
        "event_init_s": init,
        "event_completion_s": float(event["Completion"]),
        "nearest_fog_any_onset_s": nearest,
        "absolute_difference_s": error,
        "onset_within_tolerance": (
            bool(error <= MATCH_TOLERANCE_S)
            if np.isfinite(error) else False
        ),
    })

display(pd.DataFrame(comparison))

In [ ]:
# Validate event starts using class-specific label transitions.
# Do not use fog_any for this comparison.

TYPE_COLS = ["StartHesitation", "Turn", "Walking"]
TOLERANCE_S = 0.05

# Use the existing df and events variables from the previous cell.
results = []

for event_type, group in events.groupby("Type", sort=False):
    if event_type not in TYPE_COLS:
        print(f"Skipping unexpected event type: {event_type}")
        continue

    labels = df[event_type].fillna(0).astype(int).to_numpy()

    # A typed onset occurs when that class changes from 0 to 1.
    onset_indices = np.flatnonzero(
        (labels[1:] == 1) & (labels[:-1] == 0)
    ) + 1

    onset_times = df.loc[onset_indices, "time_s"].to_numpy()

    for _, event in group.iterrows():
        event_start = float(event["Init"])

        if len(onset_times) == 0:
            nearest_time = np.nan
            error = np.nan
        else:
            nearest_idx = np.argmin(np.abs(onset_times - event_start))
            nearest_time = float(onset_times[nearest_idx])
            error = abs(nearest_time - event_start)

        results.append({
            "Type": event_type,
            "event_start_s": event_start,
            "event_end_s": float(event["Completion"]),
            "nearest_typed_onset_s": nearest_time,
            "absolute_error_s": error,
            "matched_within_tolerance": (
                bool(error <= TOLERANCE_S)
                if np.isfinite(error) else False
            ),
        })

typed_alignment = pd.DataFrame(results).sort_values("event_start_s")
display(typed_alignment)

print("\nSummary by event type:")
display(
    typed_alignment.groupby("Type").agg(
        events=("Type", "size"),
        matched=("matched_within_tolerance", "sum"),
        mean_error_s=("absolute_error_s", "mean"),
        max_error_s=("absolute_error_s", "max"),
    )
)

In [ ]:

# ============================================================
# CELL 15 — DATASET-WIDE, CLASS-SPECIFIC EVENT ALIGNMENT AUDIT
# ============================================================
# Purpose:
#   1. Compare events.csv annotations with typed label transitions.
#   2. Match events to onsets one-to-one within 50 ms.
#   3. Report unmatched annotations and unmatched label onsets.
#   4. Keep notype separate because it lacks typed event labels.
#
# Does NOT modify preprocessing or forecasting labels.

from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path("/kaggle/input/competitions/tlvmc-parkinsons-freezing-gait-prediction")
TOLERANCE_S = 0.050
TYPE_COLS = ["StartHesitation", "Turn", "Walking"]

events = pd.read_csv(ROOT / "events.csv")
required_event_cols = {"Id", "Init", "Completion", "Type"}
missing = required_event_cols - set(events.columns)
if missing:
    raise ValueError(f"events.csv missing required columns: {missing}")

events = events.copy()
events["Id"] = events["Id"].astype(str)
events["Init"] = pd.to_numeric(events["Init"], errors="raise")
events["Completion"] = pd.to_numeric(events["Completion"], errors="raise")

# Locate recording files without assuming an extra train/ directory.
def locate_recording(cohort, recording_id):
    candidates = [
        ROOT / cohort / f"{recording_id}.csv",
        ROOT / "train" / cohort / f"{recording_id}.csv",
    ]
    for path in candidates:
        if path.exists():
            return path
    return None


def as_bool(series):
    """Parse common numeric and textual boolean representations safely."""
    if pd.api.types.is_bool_dtype(series):
        return series.fillna(False).to_numpy()

    if pd.api.types.is_numeric_dtype(series):
        return series.fillna(0).to_numpy() != 0

    normalized = series.astype("string").str.strip().str.lower()
    return normalized.isin(["1", "true", "t", "yes", "y"]).to_numpy()


def one_to_one_match(event_times, onset_times, tolerance):
    """
    Ordered dynamic-programming match:
      - maximize the number of matches within tolerance;
      - among those solutions, minimize total absolute timing error;
      - never assign an onset to more than one event.
    Returns list of (event_index, onset_index, absolute_error).
    """
    event_times = np.asarray(event_times, dtype=float)
    onset_times = np.asarray(onset_times, dtype=float)

    # Both inputs must be in chronological order.
    m, n = len(event_times), len(onset_times)
    if m == 0 or n == 0:
        return []

    # DP objectives: maximum match count, then minimum total error.
    count = np.zeros((m + 1, n + 1), dtype=np.int32)
    cost = np.zeros((m + 1, n + 1), dtype=float)
    action = np.zeros((m + 1, n + 1), dtype=np.int8)
    # action: 1=skip event, 2=skip onset, 3=match

    for i in range(1, m + 1):
        action[i, 0] = 1
    for j in range(1, n + 1):
        action[0, j] = 2

    for i in range(1, m + 1):
        for j in range(1, n + 1):
            # Start with skipping this event.
            best_count = count[i - 1, j]
            best_cost = cost[i - 1, j]
            best_action = 1

            # Or skip this onset.
            c2, e2 = count[i, j - 1], cost[i, j - 1]
            if c2 > best_count or (c2 == best_count and e2 < best_cost):
                best_count, best_cost, best_action = c2, e2, 2

            # Or match this event to this onset.
            error = abs(event_times[i - 1] - onset_times[j - 1])
            if error <= tolerance:
                c3 = count[i - 1, j - 1] + 1
                e3 = cost[i - 1, j - 1] + error
                if c3 > best_count or (c3 == best_count and e3 < best_cost):
                    best_count, best_cost, best_action = c3, e3, 3

            count[i, j] = best_count
            cost[i, j] = best_cost
            action[i, j] = best_action

    matches = []
    i, j = m, n
    while i > 0 and j > 0:
        step = action[i, j]
        if step == 3:
            err = abs(event_times[i - 1] - onset_times[j - 1])
            matches.append((i - 1, j - 1, err))
            i -= 1
            j -= 1
        elif step == 1:
            i -= 1
        else:
            j -= 1

    return matches[::-1]


event_rows = []
onset_rows = []
recording_rows = []
missing_files = []

# Audit every available recording in the two typed-label cohorts.
for cohort in ["tdcsfog", "defog"]:
    cohort_dir_candidates = [ROOT / cohort, ROOT / "train" / cohort]
    cohort_dir = next((p for p in cohort_dir_candidates if p.is_dir()), None)

    if cohort_dir is None:
        print(f"WARNING: directory not found for {cohort}")
        continue

    recording_paths = sorted(cohort_dir.glob("*.csv"))
    cohort_events = events[events["Id"].isin(
        [p.stem for p in recording_paths]
    )].copy()

    print(f"{cohort}: {len(recording_paths)} recording files")

    for path in recording_paths:
        recording_id = path.stem
        df = pd.read_csv(path)

        required_cols = {"Time", *TYPE_COLS}
        if cohort == "defog":
            required_cols |= {"Valid", "Task"}

        missing_cols = required_cols - set(df.columns)
        if missing_cols:
            recording_rows.append({
                "cohort": cohort,
                "Id": recording_id,
                "status": f"missing columns: {sorted(missing_cols)}",
            })
            continue

        time = pd.to_numeric(df["Time"], errors="coerce").to_numpy()
        if not np.isfinite(time).all() or (
            len(time) > 1 and not np.all(np.diff(time) > 0)
        ):
            recording_rows.append({
                "cohort": cohort,
                "Id": recording_id,
                "status": "invalid/non-increasing Time",
            })
            continue

        # Documented cohort sampling rates.
        fs = 128.0 if cohort == "tdcsfog" else 100.0
        time_s = time / fs

        # DeFOG labels are only audited where both annotation masks apply.
        if cohort == "defog":
            valid_coverage = as_bool(df["Valid"]) & as_bool(df["Task"])
        else:
            valid_coverage = np.ones(len(df), dtype=bool)

        rec_events = cohort_events[cohort_events["Id"] == recording_id]
        rec_events = rec_events[rec_events["Type"].isin(TYPE_COLS)].copy()

        rec_match_count = 0
        rec_event_count = 0
        rec_onset_count = 0

        for event_type in TYPE_COLS:
            label = pd.to_numeric(
                df[event_type], errors="coerce"
            ).to_numpy()

            # A transition is counted only when both adjacent samples
            # are inside annotated coverage. This avoids inventing an
            # onset at the edge of an invalid/unannotated region.
            onset_mask = (
                valid_coverage[1:]
                & valid_coverage[:-1]
                & (label[1:] > 0)
                & (label[:-1] <= 0)
                & np.isfinite(label[1:])
                & np.isfinite(label[:-1])
            )
            onset_indices = np.flatnonzero(onset_mask) + 1
            onset_times = time_s[onset_indices]

            typed_events = rec_events[
                rec_events["Type"] == event_type
            ].sort_values("Init")

            event_times = typed_events["Init"].to_numpy(dtype=float)
            matches = one_to_one_match(
                event_times, onset_times, TOLERANCE_S
            )

            event_to_onset = {ei: (oi, err) for ei, oi, err in matches}
            onset_to_event = {oi: ei for ei, oi, err in matches}

            for ei, (_, ev) in enumerate(typed_events.iterrows()):
                rec_event_count += 1
                matched = ei in event_to_onset

                if matched:
                    oi, error = event_to_onset[ei]
                    rec_match_count += 1
                    matched_onset = float(onset_times[oi])
                    status = "matched"
                else:
                    error = np.nan
                    matched_onset = np.nan
                    status = "unmatched_event"

                event_rows.append({
                    "cohort": cohort,
                    "Id": recording_id,
                    "Type": event_type,
                    "event_start_s": float(ev["Init"]),
                    "event_end_s": float(ev["Completion"]),
                    "matched_typed_onset_s": matched_onset,
                    "absolute_error_s": error,
                    "matched_within_tolerance": matched,
                    "status": status,
                })

            for oi, onset_time in enumerate(onset_times):
                rec_onset_count += 1
                if oi in onset_to_event:
                    ei = onset_to_event[oi]
                    onset_status = "matched"
                    matched_event_start = float(event_times[ei])
                else:
                    onset_status = "unmatched_label_onset"
                    matched_event_start = np.nan

                onset_rows.append({
                    "cohort": cohort,
                    "Id": recording_id,
                    "Type": event_type,
                    "typed_onset_s": float(onset_time),
                    "matched_event_start_s": matched_event_start,
                    "status": onset_status,
                })

        recording_rows.append({
            "cohort": cohort,
            "Id": recording_id,
            "status": "audited",
            "events_audited": rec_event_count,
            "events_matched": rec_match_count,
            "unmatched_events": rec_event_count - rec_match_count,
            "typed_onsets_detected": rec_onset_count,
        })


# ============================================================
# RESULTS + EXPORTS
# ============================================================

event_audit = pd.DataFrame(event_rows)
onset_audit = pd.DataFrame(onset_rows)
recording_audit = pd.DataFrame(recording_rows)

if event_audit.empty:
    raise RuntimeError(
        "No typed events were audited. Check the competition data paths and event IDs."
    )

event_summary = (
    event_audit.groupby(["cohort", "Type"], dropna=False)
    .agg(
        events=("Type", "size"),
        matched=("matched_within_tolerance", "sum"),
        mean_error_s=("absolute_error_s", "mean"),
        max_error_s=("absolute_error_s", "max"),
    )
    .reset_index()
)
event_summary["unmatched"] = event_summary["events"] - event_summary["matched"]
event_summary["match_rate"] = (
    event_summary["matched"] / event_summary["events"]
)

onset_summary = (
    onset_audit.groupby(["cohort", "Type", "status"], dropna=False)
    .size()
    .rename("count")
    .reset_index()
)

recording_summary = (
    recording_audit[recording_audit["status"] == "audited"]
    .groupby("cohort")
    .agg(
        recordings=("Id", "nunique"),
        recordings_with_unmatched_events=(
            "unmatched_events", lambda s: int((s > 0).sum())
        ),
        total_events=("events_audited", "sum"),
        matched_events=("events_matched", "sum"),
        total_unmatched_events=("unmatched_events", "sum"),
    )
    .reset_index()
)

OUT = Path("/kaggle/working")
event_audit.to_csv(OUT / "typed_event_alignment_details.csv", index=False)
event_summary.to_csv(OUT / "typed_event_alignment_summary.csv", index=False)
onset_audit.to_csv(OUT / "typed_onset_alignment_details.csv", index=False)
onset_summary.to_csv(OUT / "typed_onset_alignment_summary.csv", index=False)
recording_audit.to_csv(OUT / "typed_alignment_recording_audit.csv", index=False)
recording_summary.to_csv(OUT / "typed_alignment_recording_summary.csv", index=False)

print("\n=== EVENT ALIGNMENT SUMMARY (by cohort and type) ===")
display(event_summary)

print("\n=== LABEL ONSET SUMMARY ===")
display(onset_summary)

print("\n=== RECORDING SUMMARY ===")
display(recording_summary)

print("\n=== FIRST UNMATCHED EVENT ANNOTATIONS ===")
display(
    event_audit.loc[~event_audit["matched_within_tolerance"]]
    .sort_values(["cohort", "Id", "event_start_s"])
    .head(30)
)

print("\n=== FIRST UNMATCHED LABEL ONSETS ===")
display(
    onset_audit.loc[onset_audit["status"] == "unmatched_label_onset"]
    .sort_values(["cohort", "Id", "typed_onset_s"])
    .head(30)
)

print("\nSaved audit files to:", OUT)

In [ ]:

# ============================================================
# CELL 16 — DIAGNOSE UNMATCHED EVENT ANNOTATIONS
# ============================================================
# Requires: ROOT, event_audit, TYPE_COLS from Cell 15.
# Does not change labels or preprocessing.

import numpy as np
import pandas as pd

TOLERANCE_S = 0.050

unmatched = event_audit.loc[
    ~event_audit["matched_within_tolerance"]
].copy()

def find_recording(cohort, recording_id):
    candidates = [
        ROOT / cohort / f"{recording_id}.csv",
        ROOT / "train" / cohort / f"{recording_id}.csv",
    ]
    return next((p for p in candidates if p.exists()), None)

def parse_mask(series):
    if pd.api.types.is_bool_dtype(series):
        return series.fillna(False).to_numpy()
    if pd.api.types.is_numeric_dtype(series):
        return series.fillna(0).to_numpy() != 0
    return (
        series.astype("string").str.strip().str.lower()
        .isin(["1", "true", "t", "yes", "y"])
        .to_numpy()
    )

diagnostics = []

for _, ev in unmatched.iterrows():
    cohort = ev["cohort"]
    recording_id = str(ev["Id"])
    event_type = ev["Type"]
    start_s = float(ev["event_start_s"])
    end_s = float(ev["event_end_s"])

    path = find_recording(cohort, recording_id)
    if path is None:
        diagnostics.append({
            **ev.to_dict(),
            "diagnosis": "recording_file_not_found",
        })
        continue

    df = pd.read_csv(path)
    fs = 128.0 if cohort == "tdcsfog" else 100.0
    time_s = pd.to_numeric(df["Time"], errors="coerce").to_numpy() / fs
    label = pd.to_numeric(df[event_type], errors="coerce").to_numpy()

    finite = np.isfinite(time_s) & np.isfinite(label)
    if not finite.any():
        diagnostics.append({
            **ev.to_dict(),
            "diagnosis": "no_finite_time_or_label_values",
        })
        continue

    # Find the closest sample to the annotated start.
    idx = int(np.nanargmin(np.abs(time_s - start_s)))
    nearest_sample_time = float(time_s[idx])
    start_sample_error = abs(nearest_sample_time - start_s)
    start_inside_recording = (
        time_s.min() <= start_s <= time_s.max()
    )

    # Identify transitions for this class, ignoring annotation masks.
    raw_transition_mask = (
        (label[1:] > 0)
        & (label[:-1] <= 0)
        & np.isfinite(label[1:])
        & np.isfinite(label[:-1])
    )
    raw_onset_idx = np.flatnonzero(raw_transition_mask) + 1
    raw_onset_times = time_s[raw_onset_idx]

    if len(raw_onset_times):
        nearest_raw_pos = int(
            np.argmin(np.abs(raw_onset_times - start_s))
        )
        nearest_raw_onset = float(raw_onset_times[nearest_raw_pos])
        nearest_raw_error = abs(nearest_raw_onset - start_s)
    else:
        nearest_raw_onset = np.nan
        nearest_raw_error = np.nan

    # DeFOG's annotated coverage is Valid AND Task.
    if cohort == "defog":
        coverage = parse_mask(df["Valid"]) & parse_mask(df["Task"])
    else:
        coverage = np.ones(len(df), dtype=bool)

    # Count transitions that occur entirely inside annotated coverage.
    masked_transition_mask = (
        coverage[1:]
        & coverage[:-1]
        & raw_transition_mask
    )
    masked_onset_idx = np.flatnonzero(masked_transition_mask) + 1
    masked_onset_times = time_s[masked_onset_idx]

    if len(masked_onset_times):
        nearest_masked_pos = int(
            np.argmin(np.abs(masked_onset_times - start_s))
        )
        nearest_masked_onset = float(
            masked_onset_times[nearest_masked_pos]
        )
        nearest_masked_error = abs(nearest_masked_onset - start_s)
    else:
        nearest_masked_onset = np.nan
        nearest_masked_error = np.nan

    # Coverage and label statistics inside the event interval.
    interval = (time_s >= start_s) & (time_s <= end_s)
    interval_count = int(interval.sum())

    if interval_count:
        coverage_fraction = float(coverage[interval].mean())
        positive_fraction = float(
            np.mean(label[interval][np.isfinite(label[interval])] > 0)
        ) if np.isfinite(label[interval]).any() else np.nan
    else:
        coverage_fraction = np.nan
        positive_fraction = np.nan

    # Check label state at samples immediately around the event start.
    # "before" is the preceding sample; "at" is the nearest sample.
    before_idx = max(0, idx - 1)
    label_before = label[before_idx]
    label_at = label[idx]

    # Summarize the evidence without automatically assigning a cause.
    if not start_inside_recording:
        diagnosis = "event_start_outside_recording_time_range"
    elif cohort == "defog" and not coverage[idx]:
        diagnosis = "event_start_nearest_sample_outside_valid_task_coverage"
    elif np.isfinite(nearest_raw_error) and nearest_raw_error <= TOLERANCE_S:
        diagnosis = "raw_transition_exists_but_masked_transition_did_not_match"
    elif np.isfinite(label_at) and label_at > 0:
        diagnosis = "label_already_positive_at_event_start_no_nearby_onset"
    elif np.isfinite(nearest_raw_error):
        diagnosis = "class_transition_exists_but_is_too_far_from_event_start"
    else:
        diagnosis = "no_raw_class_onset_found_in_recording"

    diagnostics.append({
        "cohort": cohort,
        "Id": recording_id,
        "Type": event_type,
        "event_start_s": start_s,
        "event_end_s": end_s,
        "nearest_sample_time_s": nearest_sample_time,
        "nearest_sample_error_s": start_sample_error,
        "start_inside_recording": start_inside_recording,
        "coverage_at_nearest_sample": bool(coverage[idx]),
        "coverage_fraction_during_event": coverage_fraction,
        "label_before_start": label_before,
        "label_at_start": label_at,
        "positive_label_fraction_during_event": positive_fraction,
        "nearest_raw_onset_s": nearest_raw_onset,
        "nearest_raw_onset_error_s": nearest_raw_error,
        "nearest_masked_onset_s": nearest_masked_onset,
        "nearest_masked_onset_error_s": nearest_masked_error,
        "diagnosis": diagnosis,
    })

diagnostic_df = pd.DataFrame(diagnostics)

print("=== DIAGNOSIS COUNTS ===")
display(
    diagnostic_df.groupby(["cohort", "Type", "diagnosis"], dropna=False)
    .size()
    .rename("events")
    .reset_index()
    .sort_values(["cohort", "Type", "events"], ascending=[True, True, False])
)

print("\n=== UNMATCHED EVENTS: DETAILED RESULTS ===")
display(
    diagnostic_df.sort_values(
        ["cohort", "Id", "event_start_s"]
    )
)

diagnostic_df.to_csv(
    "/kaggle/working/unmatched_event_diagnostics.csv",
    index=False
)
print("\nSaved: /kaggle/working/unmatched_event_diagnostics.csv")

In [ ]:
# ============================================================
# Team Asha — forecasting window generation
# One pass. Stores each window ONCE with its time-to-next-onset.
# Horizon labels are derived from that, so there is no 8x duplication
# and no chance of the horizons drifting out of sync.
# ============================================================
import os, numpy as np, pandas as pd
from pathlib import Path
from scipy.signal import resample_poly

# --- point this at whichever path your notebook already uses ---
CANDIDATES = [
    Path("/kaggle/input/tlvmc-parkinsons-freezing-gait-prediction"),
    Path("/kaggle/input/competitions/tlvmc-parkinsons-freezing-gait-prediction"),
]
ROOT = next((p for p in CANDIDATES if p.exists()), None)
assert ROOT is not None, "dataset root not found — set ROOT manually"
TRAIN = ROOT / "train"
OUT = Path("/kaggle/working"); OUT.mkdir(exist_ok=True)

FS          = 100          # target rate for every cohort
WIN_S       = 4.0          # window length
STRIDE_S    = 0.5
BAND_HALF_S = 0.25         # half-width of the lead-time band
HORIZONS    = [0.5, 1.0, 1.5, 2.0, 2.5, 3.0, 4.0, 5.0]
GUARD_S     = 10.0         # negatives must be this far from any onset
NEG_PER_POS = 5            # per-recording cap, trimmed per-subject later
RNG         = np.random.default_rng(0)

WIN_N    = int(WIN_S * FS)          # 400
STRIDE_N = int(STRIDE_S * FS)       # 50
ACC      = ["AccV", "AccML", "AccAP"]

COHORTS = {
    "tdcsfog": dict(fs_src=128, to_g=True,  label_cols=["StartHesitation","Turn","Walking"], masked=False),
    "defog":   dict(fs_src=100, to_g=False, label_cols=["StartHesitation","Turn","Walking"], masked=True),
    "notype":  dict(fs_src=100, to_g=False, label_cols=["Event"],                            masked=True),
}

# ---- subject lookup -----------------------------------------
meta = []
for f, c in [("tdcsfog_metadata.csv","tdcsfog"), ("defog_metadata.csv","defog")]:
    m = pd.read_csv(ROOT / f)
    m["meta_cohort"] = c
    meta.append(m)
meta = pd.concat(meta, ignore_index=True)
SUBJ  = dict(zip(meta.Id, meta.Subject))
VISIT = dict(zip(meta.Id, meta.Visit))
MED   = dict(zip(meta.Id, meta.get("Medication", pd.Series(index=meta.index, dtype=object))))


def load_recording(path, cohort):
    """Returns accel (n,3) float32 in g at 100 Hz, onset indices, and a
    boolean eligibility mask — all in the resampled index space."""
    cfg = COHORTS[cohort]
    df = pd.read_csv(path)
    for c in ACC + cfg["label_cols"]:
        if c not in df.columns:
            raise ValueError(f"{path.name}: missing {c}")

    acc = df[ACC].to_numpy(np.float32)
    if cfg["to_g"]:
        acc = acc / 9.80665
    if not np.isfinite(acc).all():
        raise ValueError(f"{path.name}: non-finite accel")

    n_src = len(df)
    # onsets from CLASS-SPECIFIC 0->1 transitions, in source index space
    onsets = set()
    for c in cfg["label_cols"]:
        v = df[c].to_numpy()
        v = np.nan_to_num(v, nan=0).astype(np.int8)
        idx = np.flatnonzero(np.diff(np.r_[0, v]) == 1)
        onsets.update(idx.tolist())

    if cfg["masked"]:
        valid = df["Valid"].fillna(False).to_numpy(bool)
        task  = df["Task"].fillna(False).to_numpy(bool)
        elig  = task                      # negatives need annotated territory
        onsets = {i for i in onsets if task[i] and valid[i]}   # positives need both
    else:
        elig = np.ones(n_src, bool)

    # resample accel 128 -> 100 only for tdcsfog; map indices rather than
    # resampling binary columns
    if cfg["fs_src"] != FS:
        acc = resample_poly(acc, FS, cfg["fs_src"], axis=0).astype(np.float32)
        n_dst = acc.shape[0]
        scale = FS / cfg["fs_src"]
        onsets = {min(int(round(i * scale)), n_dst - 1) for i in onsets}
        src_of = np.clip((np.arange(n_dst) / scale).round().astype(int), 0, n_src - 1)
        elig = elig[src_of]
    else:
        n_dst = acc.shape[0]

    return acc, np.array(sorted(onsets), dtype=np.int64), elig


def windows_for(path, cohort):
    acc, onsets, elig = load_recording(path, cohort)
    n = acc.shape[0]
    if n < WIN_N:
        return None
    ends = np.arange(WIN_N - 1, n, STRIDE_N)                  # inclusive last sample

    # fully-eligible windows only
    ok = np.ones(len(ends), bool)
    if not elig.all():
        cum = np.r_[0, np.cumsum(elig.astype(np.int32))]
        covered = cum[ends + 1] - cum[ends + 1 - WIN_N]
        ok = covered == WIN_N
    ends = ends[ok]
    if len(ends) == 0:
        return None

    # time to NEXT onset strictly after window end, and to previous onset
    if len(onsets):
        nxt = np.searchsorted(onsets, ends, side="right")
        d_next = np.where(nxt < len(onsets), onsets[np.minimum(nxt, len(onsets)-1)] - ends, np.iinfo(np.int64).max)
        prv = nxt - 1
        d_prev = np.where(prv >= 0, ends - onsets[np.maximum(prv, 0)], np.iinfo(np.int64).max)
    else:
        d_next = np.full(len(ends), np.iinfo(np.int64).max)
        d_prev = d_next.copy()

    d_next_s = d_next / FS
    d_prev_s = d_prev / FS

    lo, hi = min(HORIZONS) - BAND_HALF_S, max(HORIZONS) + BAND_HALF_S
    is_pos = (d_next_s > lo) & (d_next_s <= hi)
    is_neg = (d_next_s > GUARD_S) & (d_prev_s > GUARD_S)

    keep_pos = np.flatnonzero(is_pos)
    neg_pool = np.flatnonzero(is_neg)
    n_neg = min(len(neg_pool), max(NEG_PER_POS * len(keep_pos), 20))
    keep_neg = RNG.choice(neg_pool, size=n_neg, replace=False) if n_neg else np.array([], int)

    keep = np.sort(np.r_[keep_pos, keep_neg])
    if len(keep) == 0:
        return None
    e = ends[keep]
    X = np.stack([acc[i - WIN_N + 1 : i + 1] for i in e]).astype(np.float16)

    rid = path.stem
    info = pd.DataFrame({
        "rec_id": rid,
        "cohort": cohort,
        "subject": SUBJ.get(rid, "UNKNOWN"),
        "visit": VISIT.get(rid, np.nan),
        "medication": MED.get(rid, np.nan),
        "end_idx": e,
        "end_s": e / FS,
        "d_next_onset_s": d_next_s[keep],
        "d_prev_onset_s": d_prev_s[keep],
        "is_pos_any": is_pos[keep],
    })
    return X, info


Xs, metas, failed = [], [], []
for cohort in COHORTS:
    files = sorted((TRAIN / cohort).glob("*.csv"))
    print(f"{cohort}: {len(files)} files")
    for j, f in enumerate(files):
        try:
            r = windows_for(f, cohort)
            if r:
                Xs.append(r[0]); metas.append(r[1])
        except Exception as ex:
            failed.append((cohort, f.name, str(ex)))
        if (j + 1) % 100 == 0:
            print(f"  {j+1}/{len(files)}")

X = np.concatenate(Xs); M = pd.concat(metas, ignore_index=True)
del Xs, metas
print("\nwindows:", X.shape, "| recordings failed:", len(failed))
print(M.groupby("cohort").is_pos_any.agg(["size", "sum"]))
if failed: print(failed[:10])

In [ ]:
# ---- trim negatives to 5:1 per SUBJECT (long recordings must not dominate)
rows = []
for s, g in M.groupby("subject"):
    pos = g.index[g.is_pos_any].to_numpy()
    neg = g.index[~g.is_pos_any].to_numpy()
    k = min(len(neg), max(NEG_PER_POS * len(pos), 50))
    rows.append(np.r_[pos, RNG.choice(neg, k, replace=False) if k else np.array([], int)])
sel = np.sort(np.concatenate(rows))
X, M = X[sel], M.iloc[sel].reset_index(drop=True)

# ---- horizon labels: lead-time BAND, not cumulative.
# window is positive for h iff the next onset lands in (h-0.25, h+0.25]
for h in HORIZONS:
    M[f"y_{h}"] = ((M.d_next_onset_s > h - BAND_HALF_S) &
                   (M.d_next_onset_s <= h + BAND_HALF_S)).astype(np.int8)
M["y_neg"] = ((M.d_next_onset_s > GUARD_S) & (M.d_prev_onset_s > GUARD_S)).astype(np.int8)

print(M[[f"y_{h}" for h in HORIZONS]].sum(), "\nnegatives:", int(M.y_neg.sum()))

np.save(OUT / "windows_X.npy", X)
M.to_parquet(OUT / "windows_meta.parquet", index=False)
print("saved", X.shape, X.nbytes / 1e9, "GB")

In [ ]:
# Hand-check the band labels against raw onsets.
# If this prints anything but True, STOP — the horizon curve would be invalid.
rid = M.loc[M.is_pos_any, "rec_id"].iloc[0]
coh = M.loc[M.rec_id == rid, "cohort"].iloc[0]
_, onsets, _ = load_recording(TRAIN / coh / f"{rid}.csv", coh)
sub = M[M.rec_id == rid]

BIG = 1e6
checks = []
for _, r in sub.head(400).iterrows():
    nxt = onsets[onsets > r.end_idx]
    true_d = (nxt[0] - r.end_idx) / FS if len(nxt) else np.inf

    # stored distance matches a fresh computation
    if np.isinf(true_d):
        checks.append(r.d_next_onset_s > BIG)
    else:
        checks.append(np.isclose(true_d, r.d_next_onset_s))

    # every horizon label matches the band rule
    for h in HORIZONS:
        expect = int((h - BAND_HALF_S) < true_d <= (h + BAND_HALF_S))
        checks.append(expect == int(r[f"y_{h}"]))

print(f"{rid}  {coh}")
print("all label checks pass:", all(checks), f"({len(checks)} assertions)")
print("no window contains its own onset:", bool((sub.d_next_onset_s > 0).all()))
print("positives per horizon in this recording:",
      {h: int(sub[f'y_{h}'].sum()) for h in HORIZONS})

In [ ]:
# ============================================================
# Daily personal-baseline profiler
# Streams the unlabeled parquet archive, computes window-level
# features, and caches one baseline vector per subject.
# ============================================================
import pyarrow.parquet as pq

DAILY_DIR = next((p for p in [ROOT/"unlabeled", ROOT/"train"/"daily", ROOT/"daily"] if p.exists()), None)
assert DAILY_DIR is not None, f"daily/unlabeled dir not found under {ROOT}"
print("daily dir:", DAILY_DIR, "| files:", len(list(DAILY_DIR.glob('*'))))

FREQ = np.fft.rfftfreq(WIN_N, 1/FS)
B_LOCO = (FREQ >= 0.5) & (FREQ < 3.0)     # locomotor band
B_FOG  = (FREQ >= 3.0) & (FREQ < 8.0)     # FoG-associated band

def window_features(W):
    """W: (n, WIN_N, 3) float32, g. Returns (n, F) float32.
    THE SAME function is used for daily baselines and for model windows —
    do not fork it, or the z-scores compare different quantities."""
    W = np.asarray(W, np.float32)
    mag = np.linalg.norm(W, axis=2)                       # (n, WIN_N)
    mag_c = mag - mag.mean(axis=1, keepdims=True)
    jerk = np.diff(mag, axis=1) * FS

    P = np.abs(np.fft.rfft(mag_c, axis=1))**2
    p_loco = P[:, B_LOCO].sum(1)
    p_fog  = P[:, B_FOG].sum(1)
    tot    = P.sum(1) + 1e-12

    f = np.stack([
        mag.mean(1),
        mag.std(1),
        np.abs(mag_c).mean(1),                            # movement intensity
        np.sqrt((jerk**2).mean(1)),                       # jerk RMS
        np.abs(jerk).max(1),
        W.std(axis=1).mean(1),                            # mean per-axis variability
        np.log10(p_loco + 1e-12),
        np.log10(p_fog + 1e-12),
        p_fog / (p_loco + 1e-12),                         # FoG-band ratio
        p_loco / tot,
    ], axis=1)
    return np.nan_to_num(f, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)

FEAT_NAMES = ["mag_mean","mag_std","mag_absdev","jerk_rms","jerk_max",
              "axis_std","logp_loco","logp_fog","fog_ratio","loco_frac"]
ACTIVE_IX = FEAT_NAMES.index("mag_absdev")
ACTIVE_THR = 0.02   # g; windows below this are treated as rest

dmeta = pd.read_csv(ROOT / "daily_metadata.csv")
need = set(M.subject.unique())                       # only subjects we model
todo = dmeta[dmeta.Subject.isin(need)]
print(f"daily recordings to profile: {len(todo)} of {len(dmeta)}")

BATCH_ROWS = WIN_N * 5000     # 2M rows ≈ manageable per batch
rows = []

for n_done, rec in enumerate(todo.itertuples(), 1):
    cand = [DAILY_DIR / f"{rec.Id}.parquet", DAILY_DIR / f"{rec.Id}.csv"]
    path = next((p for p in cand if p.exists()), None)
    if path is None:
        print("  missing:", rec.Id); continue

    feats = []
    try:
        if path.suffix == ".parquet":
            pf = pq.ParquetFile(path)
            for b in pf.iter_batches(batch_size=BATCH_ROWS, columns=ACC):
                a = np.column_stack([b.column(c).to_numpy(zero_copy_only=False) for c in ACC]).astype(np.float32)
                k = len(a) // WIN_N
                if k == 0: continue
                feats.append(window_features(a[:k*WIN_N].reshape(k, WIN_N, 3)))
        else:
            for ch in pd.read_csv(path, usecols=ACC, chunksize=BATCH_ROWS):
                a = ch.to_numpy(np.float32)
                k = len(a) // WIN_N
                if k == 0: continue
                feats.append(window_features(a[:k*WIN_N].reshape(k, WIN_N, 3)))
    except Exception as ex:
        print("  failed:", rec.Id, ex); continue

    if not feats: continue
    F = np.concatenate(feats); del feats
    active = F[:, ACTIVE_IX] > ACTIVE_THR
    Fa = F[active] if active.sum() > 100 else F     # fall back if mostly rest

    r = {"Subject": rec.Subject, "daily_id": rec.Id,
         "n_windows": len(F), "activity_fraction": float(active.mean())}
    for j, nm in enumerate(FEAT_NAMES):
        col = Fa[:, j]
        r[f"base_{nm}_mean"] = float(col.mean())
        r[f"base_{nm}_std"]  = float(col.std() + 1e-6)   # the z-score denominator
        r[f"base_{nm}_p10"]  = float(np.percentile(col, 10))
        r[f"base_{nm}_p50"]  = float(np.percentile(col, 50))
        r[f"base_{nm}_p90"]  = float(np.percentile(col, 90))
    rows.append(r)
    print(f"[{n_done}/{len(todo)}] {rec.Subject} {rec.Id}  windows={len(F):,}  active={r['activity_fraction']:.2f}")
    del F

B = pd.DataFrame(rows)
B.to_parquet(OUT / "daily_baselines.parquet", index=False)
print("\nbaselines:", B.shape, "| subjects covered:",
      B.Subject.nunique(), "of", len(need))

In [ ]:
# Window features for every modelled window, z-scored against that
# subject's own daily baseline. Subjects with no daily recording get
# deviation = 0 and a has_baseline flag of 0.
Fw = window_features(X.astype(np.float32))
Fw_df = pd.DataFrame(Fw, columns=FEAT_NAMES)

bi = B.set_index("Subject")
mu = bi[[f"base_{n}_mean" for n in FEAT_NAMES]].rename(columns=lambda c: c[5:-5])
sd = bi[[f"base_{n}_std"  for n in FEAT_NAMES]].rename(columns=lambda c: c[5:-4])

MU = mu.reindex(M.subject).to_numpy(np.float32)
SD = sd.reindex(M.subject).to_numpy(np.float32)
has = np.isfinite(MU).all(1)
MU = np.nan_to_num(MU); SD = np.where(np.isfinite(SD) & (SD > 0), SD, 1.0)

DEV = ((Fw - MU) / SD).astype(np.float32)
DEV[~has] = 0.0
M["has_baseline"] = has.astype(np.int8)

np.save(OUT / "window_feats.npy", Fw)
np.save(OUT / "window_dev.npy", DEV)
M.to_parquet(OUT / "windows_meta.parquet", index=False)
print("deviation features:", DEV.shape, "| windows with a real baseline:",
      int(has.sum()), f"({has.mean():.0%})")
print("subjects with baseline:", M.loc[has, 'subject'].nunique())

In [ ]:
# ---- restrict to subjects with a real daily baseline; A/B/C share this cohort
ABL = M.index[M.has_baseline == 1].to_numpy()
Mab = M.loc[ABL].reset_index(drop=True)
print("ablation cohort:", len(Mab), "windows |", Mab.subject.nunique(), "subjects |",
      dict(Mab.cohort.value_counts()))

# ---- clinical state (B_i, C_i, M_i)
subj = pd.read_csv(ROOT / "subjects.csv")
CLIN = ["Age","YearsSinceDx","UPDRSIII_On","UPDRSIII_Off","NFOGQ"]
subj["sex_m"] = (subj.Sex.astype(str).str.upper().str[0] == "M").astype(float)

key = Mab[["subject","visit"]].rename(columns={"subject":"Subject","visit":"Visit"})
C = key.merge(subj[["Subject","Visit","sex_m"] + CLIN], on=["Subject","Visit"], how="left")
C["med_on"] = (Mab.medication.astype(str).str.lower().str.startswith("on")).astype(float)

CLIN_COLS = ["Age","YearsSinceDx","UPDRSIII_On","UPDRSIII_Off","NFOGQ","sex_m","med_on"]
for c in CLIN_COLS:
    C[f"{c}_miss"] = C[c].isna().astype(float)
CLIN_ALL = CLIN_COLS + [f"{c}_miss" for c in CLIN_COLS]
CLINF = C[CLIN_ALL].to_numpy(np.float32)
print("clinical matrix:", CLINF.shape, "| missing per col:",
      dict(zip(CLIN_COLS, C[CLIN_COLS].isna().sum().values)))

# ---- folds over the 36 subjects only
from sklearn.model_selection import GroupKFold
N_FOLDS = 6
Mab["fold"] = -1
for k, (_, te) in enumerate(GroupKFold(n_splits=N_FOLDS).split(Mab, groups=Mab.subject)):
    Mab.loc[te, "fold"] = k
assert Mab.groupby("subject").fold.nunique().max() == 1, "SUBJECT LEAK"
print(Mab.groupby("fold").agg(windows=("subject","size"), subjects=("subject","nunique")))

Xab  = X[ABL]
DEVab = DEV[ABL]
Yab  = Mab[[f"y_{h}" for h in HORIZONS]].to_numpy(np.float32)
NEGab = Mab["y_neg"].to_numpy(np.float32)
print("\nprevalence per horizon (positives / (positives+clean negatives)):")
for j, h in enumerate(HORIZONS):
    p, n = Yab[:, j].sum(), NEGab.sum()
    print(f"  h={h}: {int(p):>5} pos, {int(n):>6} neg, prevalence {p/(p+n):.4f}")

In [ ]:
import torch, torch.nn as nn
from sklearn.metrics import average_precision_score, f1_score
dev = "cuda" if torch.cuda.is_available() else "cpu"
torch.manual_seed(0)

class Net(nn.Module):
    def __init__(self, n_dev=0, n_clin=0, nh=len(HORIZONS)):
        super().__init__()
        def blk(i, o, k, s): return nn.Sequential(
            nn.Conv1d(i, o, k, stride=s, padding=k//2), nn.BatchNorm1d(o), nn.GELU())
        self.enc = nn.Sequential(
            blk(3, 32, 7, 2), blk(32, 64, 5, 2), blk(64, 96, 5, 2), blk(96, 128, 3, 2),
            nn.AdaptiveAvgPool1d(1), nn.Flatten())
        d = 128 + n_dev + n_clin
        self.side = nn.Sequential(nn.Linear(n_dev + n_clin, 64), nn.GELU()) if (n_dev + n_clin) else None
        d = 128 + (64 if self.side else 0)
        self.head = nn.Sequential(nn.Linear(d, 128), nn.GELU(), nn.Dropout(0.3), nn.Linear(128, nh))
    def forward(self, x, s=None):
        z = self.enc(x)
        if self.side is not None:
            z = torch.cat([z, self.side(s)], 1)
        return self.head(z)

def masked_bce(logits, y, neg, pos_w):
    """Head h learns ONLY from its own positives and the clean (>10 s) negatives.
    Pre-FoG windows at other lead times are excluded from head h — otherwise the
    negative class mixes 'far from FoG' with 'near FoG, wrong lead time' and the
    degradation curve stops meaning anything."""
    m = (y > 0) | neg.unsqueeze(1).bool()
    l = nn.functional.binary_cross_entropy_with_logits(
        logits, y, weight=None, pos_weight=pos_w, reduction="none")
    return (l * m).sum() / m.sum().clamp(min=1)


In [ ]:
# ---- subject-level personalisation vector (what B should have had)
BCOLS = [f"base_{n}_mean" for n in FEAT_NAMES] + ["activity_fraction"]
bmat = B.set_index("Subject")[BCOLS]
BASE = bmat.reindex(Mab.subject).to_numpy(np.float32)
BASE = np.nan_to_num(BASE, nan=0.0)
print("baseline matrix:", BASE.shape)

def run2(mode, epochs=12, bs=256, lr=2e-3, save_oof=True):
    """A = accel | B = accel + personal baseline + deviation | C = B + clinical"""
    use_pb   = mode in ("B","C")
    use_clin = mode == "C"
    rows, OOF = [], np.full((len(Mab), len(HORIZONS)), np.nan, np.float32)
    for f in range(N_FOLDS):
        tr = np.flatnonzero(Mab.fold.to_numpy() != f); te = np.flatnonzero(Mab.fold.to_numpy() == f)
        mu = Xab[tr].astype(np.float32).mean((0,1)); sd = Xab[tr].astype(np.float32).std((0,1)) + 1e-6

        parts_tr, parts_te = [], []
        if use_pb:
            bm, bs_ = BASE[tr].mean(0), BASE[tr].std(0) + 1e-6
            parts_tr += [(BASE[tr]-bm)/bs_, np.clip(DEVab[tr], -8, 8)]
            parts_te += [(BASE[te]-bm)/bs_, np.clip(DEVab[te], -8, 8)]
        if use_clin:
            cm = np.nan_to_num(np.nanmedian(CLINF[tr], 0))
            cf = np.where(np.isnan(CLINF), cm, CLINF)
            cmu, csd = cf[tr].mean(0), cf[tr].std(0) + 1e-6
            parts_tr.append((cf[tr]-cmu)/csd); parts_te.append((cf[te]-cmu)/csd)
        S_tr = np.concatenate(parts_tr,1).astype(np.float32) if parts_tr else None
        S_te = np.concatenate(parts_te,1).astype(np.float32) if parts_te else None
        n_side = S_tr.shape[1] if S_tr is not None else 0

        pw = torch.tensor([max((NEGab[tr].sum()/max(Yab[tr,j].sum(),1))**0.5,1.0)
                           for j in range(len(HORIZONS))], dtype=torch.float32, device=dev)
        net = Net(n_dev=n_side, n_clin=0).to(dev)
        opt = torch.optim.AdamW(net.parameters(), lr=lr, weight_decay=1e-4)
        sch = torch.optim.lr_scheduler.OneCycleLR(opt, lr, epochs*max(len(tr)//bs,1))

        for ep in range(epochs):
            net.train(); perm = np.random.permutation(len(tr))
            for i in range(0, len(perm)-bs+1, bs):
                k = perm[i:i+bs]; b = tr[k]
                xb = torch.from_numpy(((Xab[b].astype(np.float32)-mu)/sd).transpose(0,2,1)).to(dev)
                sb = torch.from_numpy(S_tr[k]).to(dev) if S_tr is not None else None
                opt.zero_grad()
                masked_bce(net(xb, sb), torch.from_numpy(Yab[b]).to(dev),
                           torch.from_numpy(NEGab[b]).to(dev), pw).backward()
                opt.step(); sch.step()

        net.eval(); P=[]
        with torch.no_grad():
            for i in range(0, len(te), 1024):
                b = te[i:i+1024]
                xb = torch.from_numpy(((Xab[b].astype(np.float32)-mu)/sd).transpose(0,2,1)).to(dev)
                sb = torch.from_numpy(S_te[i:i+1024]).to(dev) if S_te is not None else None
                P.append(torch.sigmoid(net(xb, sb)).cpu().numpy())
        P = np.concatenate(P); OOF[te] = P

        for j,h in enumerate(HORIZONS):
            m = (Yab[te,j]>0) | (NEGab[te]>0)
            if Yab[te,j][m].sum() < 5: continue
            rows.append(dict(mode=mode, fold=f, horizon=h, n_pos=int(Yab[te,j][m].sum()),
                             prevalence=float(Yab[te,j][m].mean()),
                             auprc=average_precision_score(Yab[te,j][m], P[m,j])))
        print(f"  {mode} fold {f}")
    if save_oof: np.save(OUT / f"oof_{mode}.npy", OOF)
    return pd.DataFrame(rows)

R2 = pd.concat([run2(m) for m in ["A","B","C"]], ignore_index=True)
R2.to_parquet(OUT / "ablation_results_v2.parquet", index=False)
print(R2.pivot_table(index="horizon", columns="mode", values="auprc").round(4))

# ---- the decisive check: does the gain survive WITHIN subject?
H = 2.0; j = HORIZONS.index(H)
per = []
for m in ["A","B","C"]:
    O = np.load(OUT / f"oof_{m}.npy")
    for s, g in Mab.groupby("subject"):
        ix = g.index.to_numpy()
        keep = ix[(Yab[ix,j]>0) | (NEGab[ix]>0)]
        if Yab[keep,j].sum() < 10: continue
        per.append(dict(mode=m, subject=s, n_pos=int(Yab[keep,j].sum()),
                        prevalence=float(Yab[keep,j].mean()),
                        auprc=average_precision_score(Yab[keep,j], O[keep,j])))
PS = pd.DataFrame(per)
PS.to_parquet(OUT / "per_subject_h2.parquet", index=False)
print(f"\nwithin-subject AUPRC at h={H}s (n={PS.subject.nunique()} subjects):")
print(PS.groupby("mode").auprc.agg(["mean","median","std"]).round(4))
wide = PS.pivot(index="subject", columns="mode", values="auprc")
print("C beats A within-subject:", int((wide.C > wide.A).sum()), "/", len(wide))
print("subjects below 2x their own prevalence:",
      int((wide.C < 2*PS[PS['mode']=='C'].set_index('subject').prevalence).sum()))

In [ ]:
# Per-subject threshold calibration. No retraining — post-hoc on saved OOF.
# Honest framing: the twin calibrates during an onboarding period, then operates.
from sklearn.metrics import f1_score
j = HORIZONS.index(2.0)
O = np.load(OUT / "oof_A.npy")
rows = []
for s, g in Mab.groupby("subject"):
    ix = g.sort_values(["rec_id","end_idx"]).index.to_numpy()
    ix = ix[(Yab[ix,j] > 0) | (NEGab[ix] > 0)]
    if Yab[ix,j].sum() < 20: continue
    cut = int(len(ix) * 0.4)                       # first 40% = onboarding
    cal, ev = ix[:cut], ix[cut:]
    if Yab[cal,j].sum() < 5 or Yab[ev,j].sum() < 5: continue

    grid = np.quantile(O[cal,j], np.linspace(0.5, 0.999, 60))
    t_pers = grid[np.argmax([f1_score(Yab[cal,j], O[cal,j] > t) for t in grid])]
    t_glob = np.quantile(O[:,j], 1 - Yab[:,j].mean())     # one population threshold

    rows.append(dict(subject=s, n_eval=len(ev),
                     f1_global=f1_score(Yab[ev,j], O[ev,j] > t_glob),
                     f1_personal=f1_score(Yab[ev,j], O[ev,j] > t_pers)))
T = pd.DataFrame(rows)
T["gain"] = T.f1_personal - T.f1_global
print(T[["subject","n_eval","f1_global","f1_personal","gain"]].round(3).to_string(index=False))
print(f"\nmean gain {T.gain.mean():+.4f} | personal wins {int((T.gain>0).sum())}/{len(T)}")
T.to_parquet(OUT / "threshold_personalisation.parquet", index=False)

In [ ]:
# Fair comparison: BOTH thresholds F1-optimised on calibration data.
# Only difference is scope — one threshold for everyone vs one per patient.
j = HORIZONS.index(2.0)
O = np.load(OUT / "oof_A.npy")

segs = {}
for s, g in Mab.groupby("subject"):
    ix = g.sort_values(["rec_id","end_idx"]).index.to_numpy()
    ix = ix[(Yab[ix,j] > 0) | (NEGab[ix] > 0)]
    if Yab[ix,j].sum() < 20: continue
    cut = int(len(ix) * 0.4)
    cal, ev = ix[:cut], ix[cut:]
    if Yab[cal,j].sum() < 5 or Yab[ev,j].sum() < 5: continue
    segs[s] = (cal, ev)

cal_all = np.concatenate([c for c, _ in segs.values()])
grid_g = np.quantile(O[cal_all,j], np.linspace(0.50, 0.999, 120))
t_glob = grid_g[np.argmax([f1_score(Yab[cal_all,j], O[cal_all,j] > t) for t in grid_g])]
print(f"pooled-optimal global threshold: {t_glob:.4f}")

rows = []
for s, (cal, ev) in segs.items():
    grid_p = np.quantile(O[cal,j], np.linspace(0.50, 0.999, 120))
    t_pers = grid_p[np.argmax([f1_score(Yab[cal,j], O[cal,j] > t) for t in grid_p])]
    rows.append(dict(subject=s, n_eval=len(ev),
                     prevalence_eval=float(Yab[ev,j].mean()),
                     t_global=t_glob, t_personal=float(t_pers),
                     f1_global=f1_score(Yab[ev,j], O[ev,j] > t_glob),
                     f1_personal=f1_score(Yab[ev,j], O[ev,j] > t_pers)))
T = pd.DataFrame(rows); T["gain"] = T.f1_personal - T.f1_global
print(T.round(3).to_string(index=False))

nz = T[T.gain != 0]
print(f"\nmean F1  global {T.f1_global.mean():.4f} → personal {T.f1_personal.mean():.4f}")
print(f"mean gain {T.gain.mean():+.4f} | personal wins {int((T.gain>0).sum())}/{len(T)} "
      f"({int((nz.gain>0).sum())}/{len(nz)} excluding ties)")
print(f"threshold spread across patients: {T.t_personal.min():.3f} – {T.t_personal.max():.3f}")
T.to_parquet(OUT / "threshold_personalisation.parquet", index=False)

In [ ]:
# Partial pooling. λ=0 is the population threshold, λ=1 is fully personal.
# λ is chosen LEAVE-ONE-SUBJECT-OUT, so no subject influences its own λ.
LAM = np.linspace(0, 1, 21)
subs = T.subject.tolist()
tp = dict(zip(T.subject, T.t_personal))

def f1_at(s, lam):
    cal, ev = segs[s]
    t = lam * tp[s] + (1 - lam) * t_glob
    return f1_score(Yab[ev,j], O[ev,j] > t)

grid = {s: np.array([f1_at(s, l) for l in LAM]) for s in subs}
res = []
for s in subs:
    others = np.mean([grid[o] for o in subs if o != s], axis=0)
    lam_star = LAM[int(np.argmax(others))]
    res.append(dict(subject=s, lam_star=lam_star,
                    f1_shrunk=f1_at(s, lam_star), f1_global=grid[s][0]))
S = pd.DataFrame(res); S["gain"] = S.f1_shrunk - S.f1_global
print(S.round(3).to_string(index=False))
print(f"\nλ* chosen: {sorted(S.lam_star.unique())}")
print(f"mean F1  global {S.f1_global.mean():.4f} → shrunk {S.f1_shrunk.mean():.4f}")
print(f"mean gain {S.gain.mean():+.4f} | wins {int((S.gain>0).sum())}/{len(S)}")
S.to_parquet(OUT / "threshold_shrinkage.parquet", index=False)

In [ ]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_curve

plt.rcParams.update({"figure.dpi":150, "savefig.dpi":150, "font.size":10,
                     "axes.spines.top":False, "axes.spines.right":False,
                     "axes.grid":True, "grid.alpha":0.25, "savefig.bbox":"tight",
                     "figure.facecolor":"white", "axes.facecolor":"white"})
CLR = {"A":"#8a8f98", "B":"#3b7ea1", "C":"#c2541b"}
FIG = Path("/kaggle/working/figures"); FIG.mkdir(exist_ok=True)
J = HORIZONS.index(2.0)

# ============ FIG 1 — horizon degradation curve ============
fig, ax = plt.subplots(figsize=(6.2, 4.0))
for m, lbl in [("A","A · accelerometer only"),
               ("B","B · + personal baseline"),
               ("C","C · + clinical state  (submitted)")]:
    g = R2[R2["mode"]==m].groupby("horizon").auprc
    mean, sem = g.mean(), g.std()/np.sqrt(g.count())
    ax.errorbar(mean.index, mean.values, yerr=sem.values, marker="o", ms=4,
                lw=1.8 if m=="C" else 1.2, color=CLR[m], capsize=2.5, label=lbl)
prev = R2.groupby("horizon").prevalence.mean()
ax.plot(prev.index, prev.values, "--", color="#444", lw=1.2, label="chance (prevalence)")
ax.fill_between(prev.index, 0, prev.values, color="#444", alpha=0.07)
ax.set_xlabel("warning horizon before FoG onset (s)")
ax.set_ylabel("AUPRC")
ax.set_title("Forecasting performance degrades with lead time", loc="left", fontsize=11)
ax.set_ylim(0, 0.175); ax.legend(frameon=False, fontsize=8.5, loc="upper right")
ax.annotate(f"{R2[(R2['mode']=='C')&(R2.horizon==0.5)].auprc.mean()/prev.loc[0.5]:.1f}× chance",
            xy=(0.5, 0.141), xytext=(0.85, 0.162), fontsize=8.5,
            arrowprops=dict(arrowstyle="-", lw=0.7, color="#666"))
fig.savefig(FIG/"fig1_horizon_curve.png"); plt.close(fig)

# ============ FIG 2 — per-patient heterogeneity ============
P2 = PS[PS["mode"]=="C"].sort_values("auprc").reset_index(drop=True)
fig, ax = plt.subplots(figsize=(7.4, 4.0))
below = P2.auprc < 2*P2.prevalence
ax.bar(range(len(P2)), P2.auprc, color=np.where(below, "#b03a2e", "#2d6a4f"), width=0.72)
ax.plot(range(len(P2)), 2*P2.prevalence, "k_", ms=9, mew=1.4, label="2× that patient's own prevalence")
ax.set_xticks(range(len(P2))); ax.set_xticklabels(P2.subject, rotation=90, fontsize=6.5)
ax.set_ylabel(f"AUPRC at {HORIZONS[J]} s horizon"); ax.set_xlabel("patient")
ax.set_title(f"The model is not uniformly useful: {int(below.sum())} of {len(P2)} patients "
             f"fall below 2× their own event rate", loc="left", fontsize=11)
ax.legend(frameon=False, fontsize=8.5)
fig.savefig(FIG/"fig2_per_patient.png"); plt.close(fig)

# ============ FIG 3 — clinical operating point ============
O = np.load(OUT/"oof_C.npy")
msk = (Yab[:,J] > 0) | (NEGab > 0)
yt, pp = Yab[msk,J], O[msk,J]
hours = msk.sum()*STRIDE_S/3600
pr, rc, th = precision_recall_curve(yt, pp)

fig, (a1, a2) = plt.subplots(1, 2, figsize=(10.2, 3.9))
a1.plot(rc, pr, color=CLR["C"], lw=1.8)
a1.axhline(yt.mean(), ls="--", color="#444", lw=1.1)
a1.text(0.52, yt.mean()+0.004, f"chance = {yt.mean():.3f}", fontsize=8.5, color="#444")
a1.set_xlabel("recall (freezes caught)"); a1.set_ylabel("precision")
a1.set_title(f"Precision–recall, {HORIZONS[J]} s horizon", loc="left", fontsize=10.5)
a1.set_xlim(0,1)

targets, bars, labels = [0.3, 0.5, 0.7], [], []
for t in targets:
    i = np.argmin(np.abs(rc[:-1]-t))
    tp = yt.sum()*rc[i]; fp = tp*(1-pr[i])/max(pr[i],1e-9)
    bars.append(fp/hours); labels.append(f"P={pr[i]:.3f}")
a2.bar([f"{int(t*100)}%" for t in targets], bars, color="#8a5a2b", width=0.55)
for k,(b,l) in enumerate(zip(bars,labels)):
    a2.text(k, b*1.02, f"{b:.0f}/hr\n{l}", ha="center", fontsize=8.5)
a2.set_xlabel("recall target"); a2.set_ylabel("false alarms per hour of monitoring")
a2.set_title("Not yet deployable: alarm burden at usable recall", loc="left", fontsize=10.5)
a2.set_ylim(0, max(bars)*1.25)
fig.savefig(FIG/"fig3_operating_point.png"); plt.close(fig)

# ============ FIG 4 — why personalisation failed ============
fig, (b1, b2) = plt.subplots(1, 2, figsize=(10.2, 3.9),
                             gridspec_kw={"width_ratios":[1.15,1]})
Tt = T.sort_values("t_personal").reset_index(drop=True)
b1.scatter(Tt.t_personal, range(len(Tt)), s=26, color="#3b7ea1", zorder=3)
b1.axvline(t_glob, color="#b03a2e", lw=1.6, label=f"population optimum = {t_glob:.3f}")
b1.set_yticks(range(len(Tt))); b1.set_yticklabels(Tt.subject, fontsize=6.5)
b1.set_xlabel("per-patient optimal decision threshold"); b1.set_xlim(0,1.05)
b1.set_title(f"Optimal threshold varies hugely across patients "
             f"({Tt.t_personal.min():.2f}–{Tt.t_personal.max():.2f})", loc="left", fontsize=10.5)
b1.legend(frameon=False, fontsize=8.5, loc="lower right")

arms = ["feature\nconditioning\n(Model B)", "per-patient\nthreshold", "shrinkage\n(LOSO λ)"]
gains = [R2[R2["mode"]=="B"].auprc.mean() - R2[R2["mode"]=="A"].auprc.mean(),
         T.gain.mean(), S.gain.mean()]
wins  = [f"{int((R2.pivot_table(index=['fold','horizon'],columns='mode',values='auprc').eval('B-A')>0).sum())}/48",
         f"{int((T.gain>0).sum())}/{len(T)}", f"{int((S.gain>0).sum())}/{len(S)}"]
b2.axhline(0, color="#333", lw=1)
b2.bar(arms, gains, color=["#8a8f98","#8a8f98","#8a8f98"], width=0.5)
for k,(g,w) in enumerate(zip(gains,wins)):
    b2.text(k, g - 0.004 if g<0 else g + 0.002, f"{g:+.3f}\nwins {w}",
            ha="center", va="top" if g<0 else "bottom", fontsize=8.5)
b2.set_ylabel("mean change vs population model")
b2.set_title("Three personalisation mechanisms, three nulls", loc="left", fontsize=10.5)
b2.set_ylim(min(gains)*1.9, max(max(gains),0.01)*2.2)
fig.savefig(FIG/"fig4_personalisation_nulls.png"); plt.close(fig)

# ============ results summary for the README ============
lines = ["# Results — Team Asha\n",
         f"Cohort: {Mab.subject.nunique()} patients, {len(Mab):,} windows "
         f"(defog + notype; subject-grouped {N_FOLDS}-fold CV)\n",
         "## Horizon curve (AUPRC, mean over folds)\n",
         R2.pivot_table(index="horizon", columns="mode", values="auprc").round(4).to_markdown(),
         f"\nChance level (prevalence): {R2.groupby('horizon').prevalence.mean().round(4).to_dict()}\n",
         "## Submitted model\nModel C — accelerometry + personal baseline + clinical state.\n",
         "## Personalisation\n"]
for a, g, w in zip(["Feature conditioning (B vs A)","Per-patient threshold","LOSO shrinkage"], gains, wins):
    lines.append(f"- {a}: mean change {g:+.4f}, wins {w} — **null**")
lines += [f"\n## Heterogeneity\n- {int(below.sum())}/{len(P2)} patients below 2× their own event rate "
          f"at {HORIZONS[J]} s\n- Per-patient optimal threshold spans "
          f"{Tt.t_personal.min():.3f}–{Tt.t_personal.max():.3f} vs population {t_glob:.3f}\n",
          f"\n## Operating point ({HORIZONS[J]} s)\n"]
for t, b, l in zip(targets, bars, labels):
    lines.append(f"- recall {t:.0%} → {l}, ~{b:.0f} false alarms/hour\n")
(Path("/kaggle/working/RESULTS.md")).write_text("".join(lines))
print("".join(lines))
print("\nfigures:", [p.name for p in sorted(FIG.glob("*.png"))])

In [ ]:
# ============================================================
# Dashboard data export  ->  twin_data.json
# ============================================================
import json

J = HORIZONS.index(2.0)
O = np.load(OUT/"oof_C.npy")
tper = dict(zip(T.subject, T.t_personal))
pauc = PS[PS["mode"]=="C"].set_index("subject")

roster = []
for s, g in Mab.groupby("subject"):
    row = C.loc[g.index[0]]
    b = B[B.Subject == s]
    roster.append({
        "subject": s,
        "visit": int(g.visit.iloc[0]) if pd.notna(g.visit.iloc[0]) else None,
        "age": None if pd.isna(row.Age) else float(row.Age),
        "sex": "M" if row.sex_m == 1 else "F",
        "years_dx": None if pd.isna(row.YearsSinceDx) else float(row.YearsSinceDx),
        "updrs_on": None if pd.isna(row.UPDRSIII_On) else float(row.UPDRSIII_On),
        "updrs_off": None if pd.isna(row.UPDRSIII_Off) else float(row.UPDRSIII_Off),
        "nfogq": None if pd.isna(row.NFOGQ) else float(row.NFOGQ),
        "med_on": bool(row.med_on),
        "n_windows": int(len(g)),
        "event_rate": float(Yab[g.index.to_numpy(), J].mean()),
        "auprc": float(pauc.auprc.get(s)) if s in pauc.index else None,
        "threshold_personal": float(tper[s]) if s in tper else None,
        "activity_fraction": float(b.activity_fraction.iloc[0]) if len(b) else None,
        "baseline": {n: float(b[f"base_{n}_mean"].iloc[0]) for n in FEAT_NAMES} if len(b) else None,
    })

MAXPTS = 1200
traces = {}
for s, g in Mab.groupby("subject"):
    rid = g.rec_id.value_counts().idxmax()
    gg = g[g.rec_id == rid].sort_values("end_s")
    ix = gg.index.to_numpy()
    step = max(1, len(ix)//MAXPTS)
    k = ix[::step]
    traces[s] = {
        "rec_id": rid,
        "t":     [round(float(x),2) for x in Mab.loc[k,"end_s"]],
        "risk":  [round(float(x),4) for x in O[k, J]],
        "label": [int(x) for x in Yab[k, J]],
        "dev":   [round(float(x),3) for x in np.clip(DEVab[k][:, FEAT_NAMES.index("jerk_rms")], -6, 6)],
    }

hz = R2.pivot_table(index="horizon", columns="mode", values="auprc").round(4)
pv = R2.groupby("horizon").prevalence.mean().round(4)
op = [{"recall": t, "precision": float(l.split("=")[1]), "fa_per_hour": round(float(b_),1)}
      for t, b_, l in zip(targets, bars, labels)]

payload = {
    "meta": {
        "team": "Team Asha", "institution": "Sir MVIT, Bengaluru",
        "n_patients": int(Mab.subject.nunique()), "n_windows": int(len(Mab)),
        "horizon_s": float(HORIZONS[J]), "window_s": WIN_S, "fs": FS,
        "folds": int(N_FOLDS), "threshold_global": float(t_glob),
        "submitted_model": "C - accelerometry + personal baseline + clinical state",
    },
    "horizons": [float(h) for h in HORIZONS],
    "horizon_curve": {m: [float(hz.loc[h, m]) for h in HORIZONS] for m in ["A","B","C"]},
    "prevalence": [float(pv.loc[h]) for h in HORIZONS],
    "nulls": [{"mechanism": a, "delta": round(float(g),4), "wins": w}
              for a, g, w in zip(["Feature conditioning (B vs A)", "Per-patient threshold",
                                  "LOSO shrinkage"], gains, wins)],
    "operating_points": op,
    "heterogeneity": {
        "below_2x": int(below.sum()), "n_evaluated": int(len(P2)),
        "threshold_min": float(T.t_personal.min()), "threshold_max": float(T.t_personal.max()),
    },
    "feature_names": FEAT_NAMES,
    "patients": roster,
    "traces": traces,
}

p = OUT/"twin_data.json"
p.write_text(json.dumps(payload))
print("wrote", p, f"{p.stat().st_size/1e6:.2f} MB | patients {len(roster)} | traces {len(traces)}")

# ---- bundle every deliverable into one zip for easy download
import shutil, os
BUNDLE = OUT/"asha_outputs"
if BUNDLE.exists(): shutil.rmtree(BUNDLE)
BUNDLE.mkdir()
WANT = ["twin_data.json","RESULTS.md","ablation_results_v2.parquet","per_subject_h2.parquet",
        "threshold_personalisation.parquet","threshold_shrinkage.parquet",
        "daily_baselines.parquet","windows_meta.parquet","window_dev.npy",
        "oof_A.npy","oof_B.npy","oof_C.npy",
        "event_alignment_audit.csv","unmatched_event_diagnostics.csv"]
for n in WANT:
    f = OUT/n
    if f.exists(): shutil.copy(f, BUNDLE/n)
figdir = OUT/"figures"
if figdir.exists():
    for f in figdir.glob("*.png"): shutil.copy(f, BUNDLE/f.name)
shutil.make_archive(str(OUT/"asha_outputs"), "zip", BUNDLE)
shutil.rmtree(BUNDLE)
z = OUT/"asha_outputs.zip"
print("bundle:", z, f"{z.stat().st_size/1e6:.1f} MB")
